<a href="https://colab.research.google.com/github/MirzaBurhan/Langchain-/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-groq
!pip install youtube-transcript-api
!pip install chromadb
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install python-dotenv

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("check if the api key is loaded: ", bool(os.environ["GROQ_API_KEY"]))

In [ ]:
!pip install pytube

In [ ]:
# !pip install -qU  youtube-transcript-api
from langchain_community.document_loaders import YoutubeLoader

# Load transcript from a video URL
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=kCc8FmEb1nY&t=4595s",
    add_video_info=False  # Adds video metadata like title, author, etc.
)

docs = loader.load()
print(docs[0].page_content)
transcript= str(docs)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
# texts = text_splitter.split_text(document)

def split_transcript(transcript: str):
    """
    Split transcript into overlapping chunks
    Why overlap? → preserves context at boundaries
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # each chunk = 1000 chars
        chunk_overlap=200,    # 200 char overlap between chunks
        separators=["\n\n", "\n", ".", " "]  # split priority
    )

    chunks = splitter.create_documents([transcript])
    print(f"✅ Created {len(chunks)} chunks")
    print(f"Sample chunk:\n{chunks[0].page_content[:200]}")
    return chunks

chunks = split_transcript(transcript)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def load_embeddings():
    """
    Load free HuggingFace embedding model
    all-MiniLM-L6-v2 → fast, small, good quality ✅
    """
    print("Loading embedding model...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("✅ Embeddings model loaded!")
    return embeddings

embeddings = load_embeddings()

# Test embeddings
test_embed = embeddings.embed_query("What is a neural network?")
# print(type(test_embed)
print(f"Embedding dimensions: {len(test_embed)}")
# → 384 dimensions

In [ ]:
len(test_embed)

In [ ]:
from langchain_community.vectorstores import Chroma

def create_vector_store(chunks, embeddings):
    """
    Store chunks + embeddings in ChromaDB
    ChromaDB = local vector database (no API needed)
    """
    print("Creating vector store...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./karpathy_rag_db",  # save locally
        collection_name="karpathy_lectures"
    )

    print(f"✅ Stored {len(chunks)} chunks in ChromaDB")
    return vectorstore

vectorstore = create_vector_store(chunks, embeddings)

# Test similarity search
results = vectorstore.similarity_search(
    "what is a positional encoding?",
    k=3   # return top 3 chunks
)
print("\n🔍 Top 3 relevant chunks:")
for i, r in enumerate(results):
    print(f"\nChunk {i+1}:\n{r.page_content[:100]}")

In [ ]:
from langchain_groq import ChatGroq
def load_llm():
  llm = ChatGroq(
      model_name="llama-3.1-8b-instant",
      api_key=os.environ["GROQ_API_KEY"] ,
      temperature = 0,
      max_tokens = 1024
  )

  return llm

llm = load_llm()
response = llm.invoke("What is RAG in one sentence?")

print(response.content)



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def build_rag_chain(vectorstore, llm):
    """
    Build complete RAG pipeline using LCEL
    """

    # Step 1 — Create retriever
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 4}  # retrieve top 4 chunks
    )

    # Step 2 — Create prompt
    prompt = ChatPromptTemplate.from_template("""
    You are an AI assistant answering questions about
    Andrej Karpathy's lecture.

    Use ONLY the context below to answer.
    If the answer is not in the context, say
    "I don't have information about that in this lecture."

    Context:
    {context}

    Question: {question}

    Answer:
    """)

    # Step 3 — Format retrieved docs
    def format_docs(docs):
        return "\n\n".join([
            f"[Chunk {i+1}]: {doc.page_content}"
            for i, doc in enumerate(docs)
        ])

    # Step 4 — Build chain using LCEL
    rag_chain = (
        {
            "context":  retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain, retriever

rag_chain, retriever = build_rag_chain(vectorstore, llm)

In [ ]:
retriever

In [ ]:
def ask(question: str, show_sources: bool = True):
    """
    Ask any question about Karpathy's lecture
    """
    print(f"\n❓ Question: {question}")
    print("─" * 50)

    # Get answer
    answer = rag_chain.invoke(question)
    print(f"💡 Answer:\n{answer}")

    # Show source chunks
    if show_sources:
        docs = retriever.invoke(question)
        print(f"\n📚 Sources ({len(docs)} chunks used):")
        for i, doc in enumerate(docs):
            print(f"\n[Source {i+1}]: {doc.page_content[:150]}...")

    return answer

# Test with Karpathy lecture questions
ask("What is a large language model?")
ask("How does attention mechanism work?")
ask("What is a bigram model?")
ask("How many parameters does GPT have?")
ask("What is tokenization?")

In [24]:
# import os
# from google.colab import userdata
# from langchain_groq import ChatGroq
# def load_llm():
#   llm = ChatGroq(
#       model_name="llama-3.1-8b-instant",
#       groq_api_key=userdata.get("GROQ_API_KEY") ,
#       temperature = 0,
#       max_tokens = 1024
#   )

#   return llm

# llm = load_llm()
# response = llm.invoke("What is RAG in one sentence?")

# print(response.content)

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY']=userdata.get('GROQ_API_KEY')